# 02_Bronze_Full_Ingestion

In [0]:
from pyspark.sql.functions import current_timestamp, col

# 1. Receive Parameters from ADF
dbutils.widgets.text("p_dataset_name", "")
dbutils.widgets.text("p_source_path", "")

dataset_name = dbutils.widgets.get("p_dataset_name")
source_path = dbutils.widgets.get("p_source_path") 

catalog_name = "dbw_atlas_dev_7405606293032023"
storage_account = "statlasdev002"

# 2. Construct Paths (Using External Location)
full_source_path = f"abfss://landing@{storage_account}.dfs.core.windows.net/{dataset_name}/"
external_table_path = f"abfss://bronze@{storage_account}.dfs.core.windows.net/{dataset_name}"

# 3. Read Full CSV (Standard Spark Read, NOT Auto Loader)
df_raw = (spark.read
    .format("csv")
    .option("header", "true")
    .option("inferSchema", "true")
    .load(full_source_path)
)

# 4. Add Audit Columns
df_enriched = df_raw \
    .withColumn("_ingestion_timestamp", current_timestamp()) \
    .withColumn("_source_file_path", col("_metadata.file_path"))

# 5. Write to Unity Catalog Bronze Table (Overwrite Mode)
(df_enriched.write
    .format("delta")
    .mode("overwrite")
    .option("mergeSchema", "true")
    .option("path", external_table_path)
    .saveAsTable(f"{catalog_name}.bronze.{dataset_name}")
)

In [0]:
try:
    row_processed = df_enriched.count()
except:
    row_processed =0
dbutils.notebook.exit(str(row_processed))